<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания № 18 — Система отзывов


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Проект продолжает вариант № 18 «Система отзывов» из прошлого спринта.
Сохранены классы `Review`, `ServiceReview`, `ProductReview`, их конструкторы,
свойства и методы, включая взаимодействие объектов через `CompareRating`.

#### Дополнительное задание текущего спринта

В каждый существующий класс добавлены четыре свойства и четыре метода.
Добавлен класс `DigitalProductReview` с четырьмя собственными свойствами и методами.

| Класс | Добавленные свойства | Добавленные методы |
|---|---|---|
| Review | Author, CreatedAt, HelpfulVotes, IsPublished | EditText, Publish, Unpublish, MarkHelpful |
| ServiceReview | ServiceName, ProviderName, Cost, IsRecommended | UpdateServiceInfo, Recommend, CalculateDiscountPrice, GetDaysSinceVisit |
| ProductReview | ProductName, Brand, PurchasePrice, IsVerifiedPurchase | VerifyPurchase, UpdateProductInfo, CalculateDiscountPrice, GetPurchaseSummary |
| DigitalProductReview | Platform, Version, DownloadSizeMb, IsLicenseActivated | ActivateLicense, UpdateVersion, GetDownloadSizeGb, GetReviewDetails |

**Простое наследование:** `ServiceReview : Review`, `ProductReview : Review`.

**Многоуровневое наследование:** `Review → ProductReview → DigitalProductReview`.

**Множественное наследование через интерфейсы:** `ProductReview` реализует
`IVerifiable` и `IDiscountable`, а `ServiceReview` — `IRecommendable` и `IDiscountable`.
В C# нельзя наследоваться от нескольких классов, поэтому используются несколько интерфейсов.
`DigitalProductReview` получает реализации интерфейсов от `ProductReview`.

Геттеры и сеттеры управляют доступом к состоянию: рейтинг проверяется на диапазон 1–5,
а счётчик полезности, статус публикации и подтверждение покупки изменяются методами.
Подтверждение покупки в этом учебном проекте моделируется вызовом метода без внешнего сервиса.
Цена со скидкой рассчитывается без изменения исходной стоимости.
Для размера загрузки используются десятичные единицы: 1000 МБ = 1 ГБ.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;


// Один класс может реализовать несколько интерфейсов.
public interface IVerifiable { void VerifyPurchase(); }
public interface IDiscountable { decimal CalculateDiscountPrice(decimal percent); }
public interface IRecommendable { void Recommend(bool recommended); }

// Базовый класс Review
public class Review
{
    public int ReviewId { get; set; }
    public string Text { get; set; }
    
private int rating;
    public int Rating
    {
        get { return rating; }
        protected set
        {
            if (value < 1 || value > 5)
                throw new ArgumentOutOfRangeException(nameof(Rating), "Рейтинг должен быть от 1 до 5.");
            rating = value;
        }
    }

    // Четыре новых свойства базового класса.
    public string Author { get; set; } = "Аноним";
    public DateTime CreatedAt { get; private set; } = DateTime.Now;
    public int HelpfulVotes { get; private set; }
    public bool IsPublished { get; private set; }

    // Четыре новых метода базового класса.
    public void EditText(string text)
    {
        if (string.IsNullOrWhiteSpace(text))
            throw new ArgumentException("Текст отзыва не должен быть пустым.", nameof(text));
        Text = text;
    }
    public void Publish() { IsPublished = true; }
    public void Unpublish() { IsPublished = false; }
    public void MarkHelpful() { HelpfulVotes++; }


    public Review(int reviewId, string text, int rating)
    {
        ReviewId = reviewId;
        Text = text;
        Rating = rating;
    }

    public virtual void DisplayReview()
    {
        Console.WriteLine($"Отзыв №{ReviewId}: {Text}. Рейтинг: {Rating}/5");
    }

    public virtual void RateProduct(int rating)
    {
        Rating = rating;
        Console.WriteLine($"Рейтингу присвоено значение: {Rating}/5");
    }

    public virtual string GetReviewDetails()
    {
        return $"ID отзыва: {ReviewId}; Текст: {Text}; Рейтинг: {Rating}/5";
    }

    // Взаимодействие двух объектов Review
    public void CompareRating(Review otherReview)
    {
        if (otherReview == null) throw new ArgumentNullException(nameof(otherReview));
        Console.WriteLine($"Сравнение отзывов №{ReviewId} и №{otherReview.ReviewId}:");

        if (Rating > otherReview.Rating)
            Console.WriteLine($"У отзыва №{ReviewId} рейтинг выше");
        else if (Rating < otherReview.Rating)
            Console.WriteLine($"У отзыва №{otherReview.ReviewId} рейтинг выше");
        else
            Console.WriteLine("Рейтинги одинаковые");
    }
}

// Производный класс для отзыва об услуге
public class ServiceReview : Review, IRecommendable, IDiscountable
{
    public DateTime VisitDate { get; set; }

    // Четыре новых свойства отзыва об услуге.
    public string ServiceName { get; private set; } = "Не указана";
    public string ProviderName { get; private set; } = "Не указан";
    public decimal Cost { get; private set; }
    public bool IsRecommended { get; private set; }

    // Четыре новых метода.
    public void UpdateServiceInfo(string serviceName, string providerName, decimal cost)
    {
        if (cost < 0) throw new ArgumentOutOfRangeException(nameof(cost));
        ServiceName = serviceName; ProviderName = providerName; Cost = cost;
    }
    public void Recommend(bool recommended) { IsRecommended = recommended; }
    public decimal CalculateDiscountPrice(decimal percent)
    {
        if (percent < 0 || percent > 100) throw new ArgumentOutOfRangeException(nameof(percent));
        return Cost * (1 - percent / 100m);
    }
    public int GetDaysSinceVisit(DateTime currentDate)
    {
        if (currentDate.Date < VisitDate.Date)
            throw new ArgumentException("Дата расчёта раньше посещения.", nameof(currentDate));
        return (currentDate.Date - VisitDate.Date).Days;
    }


    public ServiceReview(int reviewId, string text, int rating, DateTime visitDate)
        : base(reviewId, text, rating)
    {
        VisitDate = visitDate;
    }

    public override void DisplayReview()
    {
        Console.WriteLine(
            $"Отзыв об услуге №{ReviewId}: {Text}. " +
            $"Рейтинг: {Rating}/5. Дата посещения: {VisitDate:dd.MM.yyyy}");
    }

    public override string GetReviewDetails()
    {
        return base.GetReviewDetails() +
               $"; Дата посещения: {VisitDate:dd.MM.yyyy}; услуга: {ServiceName}; " +
               $"исполнитель: {ProviderName}; стоимость: {Cost}; рекомендована: {IsRecommended}";
    }
}

// Производный класс для отзыва о товаре
public class ProductReview : Review, IVerifiable, IDiscountable
{
    public int ProductId { get; set; }

    // Четыре новых свойства отзыва о товаре.
    public string ProductName { get; private set; } = "Не указан";
    public string Brand { get; private set; } = "Не указан";
    public decimal PurchasePrice { get; private set; }
    public bool IsVerifiedPurchase { get; private set; }

    // Четыре новых метода.
    public void VerifyPurchase() { IsVerifiedPurchase = true; }
    public void UpdateProductInfo(string productName, string brand, decimal price)
    {
        if (price < 0) throw new ArgumentOutOfRangeException(nameof(price));
        ProductName = productName; Brand = brand; PurchasePrice = price;
    }
    public decimal CalculateDiscountPrice(decimal percent)
    {
        if (percent < 0 || percent > 100) throw new ArgumentOutOfRangeException(nameof(percent));
        return PurchasePrice * (1 - percent / 100m);
    }
    public string GetPurchaseSummary()
    {
        return $"{ProductName}, {Brand}; цена: {PurchasePrice}; подтверждена: {IsVerifiedPurchase}";
    }


    public ProductReview(int reviewId, string text, int rating, int productId)
        : base(reviewId, text, rating)
    {
        ProductId = productId;
    }

    public override void RateProduct(int rating)
    {
        Rating = rating;
        Console.WriteLine($"Товару с ID {ProductId} присвоен рейтинг: {Rating}/5");
    }

    public override string GetReviewDetails()
    {
        return base.GetReviewDetails() +
               $"; ID продукта: {ProductId}; {GetPurchaseSummary()}";
    }
}

// Многоуровневая цепочка: Review -> ProductReview -> DigitalProductReview.
public class DigitalProductReview : ProductReview
{
    public string Platform { get; set; }
    public string Version { get; private set; }
    public double DownloadSizeMb { get; private set; }
    public bool IsLicenseActivated { get; private set; }

    public DigitalProductReview(int reviewId, string text, int rating, int productId,
        string platform, string version, double downloadSizeMb)
        : base(reviewId, text, rating, productId)
    {
        if (double.IsNaN(downloadSizeMb) || double.IsInfinity(downloadSizeMb) || downloadSizeMb < 0)
            throw new ArgumentOutOfRangeException(nameof(downloadSizeMb));
        if (string.IsNullOrWhiteSpace(version)) throw new ArgumentException("Укажите версию.", nameof(version));
        Platform = platform; Version = version; DownloadSizeMb = downloadSizeMb;
    }

    public void ActivateLicense()
    {
        if (!IsVerifiedPurchase)
            throw new InvalidOperationException("Сначала подтвердите покупку.");
        IsLicenseActivated = true;
    }
    public void UpdateVersion(string version)
    {
        if (string.IsNullOrWhiteSpace(version)) throw new ArgumentException("Укажите версию.", nameof(version));
        Version = version;
    }
    public double GetDownloadSizeGb() { return DownloadSizeMb / 1000.0; }
    public override string GetReviewDetails()
    {
        return base.GetReviewDetails() +
            $"; платформа: {Platform}; версия: {Version}; размер: {DownloadSizeMb} МБ; " +
            $"лицензия активирована: {IsLicenseActivated}";
    }
}


In [ ]:
var serviceReview = new ServiceReview(
    1,
    "Хорошее обслуживание",
    5,
    new DateTime(2026, 9, 9));

var productReview = new ProductReview(
    2,
    "Качественный товар",
    4,
    101);

serviceReview.DisplayReview();
productReview.DisplayReview();

Console.WriteLine();
Console.WriteLine(serviceReview.GetReviewDetails());
Console.WriteLine(productReview.GetReviewDetails());

Console.WriteLine();
// Один объект взаимодействует с другим объектом
serviceReview.CompareRating(productReview);

Console.WriteLine();
productReview.RateProduct(5);
serviceReview.CompareRating(productReview);


Console.WriteLine("\nДополнительные свойства и методы:");
serviceReview.Author = "Дмитрий";
serviceReview.EditText("Вежливое обслуживание и быстрый ремонт");
serviceReview.UpdateServiceInfo("Ремонт ноутбука", "Сервис Плюс", 3000m);
serviceReview.Recommend(true);
serviceReview.Publish();
serviceReview.MarkHelpful();
Console.WriteLine(serviceReview.GetReviewDetails());
Console.WriteLine($"Автор: {serviceReview.Author}; создан: {serviceReview.CreatedAt:dd.MM.yyyy}; " +
    $"полезность: {serviceReview.HelpfulVotes}; опубликован: {serviceReview.IsPublished}");
Console.WriteLine($"Дней после посещения: {serviceReview.GetDaysSinceVisit(new DateTime(2026, 10, 20))}");
serviceReview.Unpublish();
Console.WriteLine($"После снятия с публикации: {serviceReview.IsPublished}");

productReview.UpdateProductInfo("Клавиатура", "Logitech", 5000m);
productReview.VerifyPurchase();
Console.WriteLine(productReview.GetPurchaseSummary());

var digitalReview = new DigitalProductReview(
    3, "Удобная программа", 4, 202, "Windows", "1.0", 1500);
digitalReview.UpdateProductInfo("Редактор фотографий", "PhotoSoft", 2000m);
digitalReview.Author = "Анна";
digitalReview.VerifyPurchase();
digitalReview.ActivateLicense();
digitalReview.UpdateVersion("1.1");
digitalReview.Publish();
Console.WriteLine($"Размер загрузки: {digitalReview.GetDownloadSizeGb()} ГБ");

// Полиморфизм: в массиве базового типа находятся разные производные объекты.
Review[] reviews = { serviceReview, productReview, digitalReview };
foreach (Review review in reviews)
    Console.WriteLine(review.GetReviewDetails());
digitalReview.CompareRating(productReview);

// Работа через интерфейсы: один объект поддерживает несколько контрактов.
IVerifiable verifiable = digitalReview;
verifiable.VerifyPurchase();
IRecommendable recommendable = serviceReview;
recommendable.Recommend(false);
Console.WriteLine($"Рекомендация после изменения: {serviceReview.IsRecommended}");
IDiscountable[] discountedItems = { serviceReview, productReview, digitalReview };
foreach (IDiscountable item in discountedItems)
    Console.WriteLine($"Стоимость со скидкой 10%: {item.CalculateDiscountPrice(10)}");

// Ошибочные данные отклоняются, а прежний рейтинг сохраняется.
try { productReview.RateProduct(6); }
catch (ArgumentOutOfRangeException) { Console.WriteLine("Рейтинг вне диапазона 1–5 отклонён."); }
Console.WriteLine($"Рейтинг после проверки: {productReview.Rating}/5");


### Запуск

Выберите ядро `.NET (C#)` и выполните две ячейки сверху вниз. Первая объявляет классы и интерфейсы, вторая демонстрирует их работу.
